# Brain Tumor Classification — EfficientNetB0
## Google Colab Training Notebook
### PCCOE MDM Capstone 2025-26
**Steps:**
1. Mount Google Drive
2. Upload dataset to Drive OR download from Kaggle
3. Install dependencies
4. Clone/upload project code
5. Train EfficientNetB0 classifier
6. Evaluate and download model

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Option A: Download dataset from Kaggle
You need a Kaggle API key. Go to kaggle.com → Account → Create API Token → download kaggle.json

In [ ]:
# Upload your kaggle.json first using the Files panel on the left
from google.colab import files
import os

# Upload kaggle.json
uploaded = files.upload()  # select your kaggle.json

# Setup kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
print("Kaggle credentials set up successfully")

In [ ]:
# Download Brain Tumor MRI Dataset
os.system('kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content/data/')
os.system('unzip -q /content/data/brain-tumor-mri-dataset.zip -d /content/data/Brain\ Tumor\ MRI\ Dataset')
print("Dataset downloaded and extracted")

# Verify structure
import os
base = '/content/data/Brain Tumor MRI Dataset'
for split in ['Training', 'Testing']:
    for cls in ['glioma', 'meningioma', 'notumor', 'pituitary']:
        path = os.path.join(base, split, cls)
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f"{split}/{cls}: {count} images")

## Install Dependencies

In [ ]:
!pip install -q tensorflow==2.17.0 opencv-python-headless==4.10.0.84 albumentations==1.4.11 scikit-learn==1.5.1 matplotlib==3.9.2 seaborn==0.13.2 tqdm==4.66.4
print("All dependencies installed")

## Setup Project Code
Either clone from GitHub or copy code cells below

In [ ]:
# Clone your GitHub repo
!git clone https://github.com/mitali2744/Brain-tumor-diagnostic-system.git /content/project
import sys
sys.path.insert(0, '/content/project')
print("Project code loaded")

In [ ]:
# Override config paths to point to Colab data directory
import os
os.environ['COLAB_ENV'] = '1'

# Patch config for Colab paths
import config
config.DATA_DIR = '/content/data'
config.CLASSIFICATION_DATA_DIR = '/content/data/Brain Tumor MRI Dataset'
config.MODELS_DIR = '/content/models'
config.CLASSIFIER_MODEL_PATH = '/content/models/efficientnetb0_classifier.h5'

os.makedirs(config.MODELS_DIR, exist_ok=True)
print("Config patched for Colab")
print(f"Classification data: {config.CLASSIFICATION_DATA_DIR}")
print(f"Models dir: {config.MODELS_DIR}")

## EDA — Quick Data Check

In [ ]:
import matplotlib.pyplot as plt
import os

base = '/content/data/Brain Tumor MRI Dataset/Training'
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']
counts = [len(os.listdir(os.path.join(base, c))) for c in classes]

plt.figure(figsize=(8, 4))
bars = plt.bar(classes, counts, color=['#e74c3c','#f39c12','#27ae60','#8e44ad'])
plt.title('Training Set Class Distribution', fontsize=13)
plt.ylabel('Number of Images')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(count), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150)
plt.show()
print("Class counts:", dict(zip(classes, counts)))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, cls in enumerate(classes):
    cls_dir = os.path.join(base, cls)
    imgs = os.listdir(cls_dir)[:2]
    for j, img_name in enumerate(imgs):
        img = cv2.imread(os.path.join(cls_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[j][i].imshow(img)
        axes[j][i].set_title(f'{cls}', fontsize=10)
        axes[j][i].axis('off')
plt.suptitle('Sample MRI Images per Class', fontsize=14)
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=150)
plt.show()

## Phase 1 Training — Freeze Base, Train Head Only

In [ ]:
import tensorflow as tf
from src.classification.efficientnet_model import build_classifier, compile_phase1, compile_phase2, unfreeze_top_layers
from src.classification.dataset import build_classification_generators

# Load data
print("Loading dataset...")
train_gen, val_gen, test_gen, class_weights = build_classification_generators()

# Build model
print("Building EfficientNetB0...")
model, base_model = build_classifier()
model.summary()

# Callbacks
import config
checkpoint_p1 = tf.keras.callbacks.ModelCheckpoint(
    '/content/models/classifier_phase1_best.h5',
    monitor='val_accuracy', save_best_only=True, verbose=1
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
)

# Phase 1 training
model = compile_phase1(model)
print("\nPhase 1: Training classification head (base frozen)...")
history_p1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    class_weight=class_weights,
    callbacks=[checkpoint_p1, early_stop, reduce_lr],
    verbose=1
)
print("Phase 1 complete")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_p1.history['accuracy'], label='Train', color='steelblue')
axes[0].plot(history_p1.history['val_accuracy'], label='Val', color='tomato')
axes[0].set_title('Phase 1 Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history_p1.history['loss'], label='Train', color='steelblue')
axes[1].plot(history_p1.history['val_loss'], label='Val', color='tomato')
axes[1].set_title('Phase 1 Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Phase 1 — Head Training')
plt.tight_layout()
plt.savefig('/content/phase1_history.png', dpi=150)
plt.show()

## Phase 2 Training — Fine-tune Top 20 Layers

In [ ]:
checkpoint_p2 = tf.keras.callbacks.ModelCheckpoint(
    '/content/models/classifier_phase2_best.h5',
    monitor='val_accuracy', save_best_only=True, verbose=1
)
early_stop2 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)
reduce_lr2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
)

unfreeze_top_layers(base_model, num_layers_to_unfreeze=20)
model = compile_phase2(model)

print("\nPhase 2: Fine-tuning top 20 layers...")
history_p2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    class_weight=class_weights,
    callbacks=[checkpoint_p2, early_stop2, reduce_lr2],
    verbose=1
)
print("Phase 2 complete")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_p2.history['accuracy'], label='Train', color='steelblue')
axes[0].plot(history_p2.history['val_accuracy'], label='Val', color='tomato')
axes[0].set_title('Phase 2 Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history_p2.history['loss'], label='Train', color='steelblue')
axes[1].plot(history_p2.history['val_loss'], label='Val', color='tomato')
axes[1].set_title('Phase 2 Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Phase 2 — Fine-tuning')
plt.tight_layout()
plt.savefig('/content/phase2_history.png', dpi=150)
plt.show()

In [ ]:
# Save final model
model.save('/content/models/efficientnetb0_classifier.h5')
print("Model saved to /content/models/efficientnetb0_classifier.h5")

# Evaluate on test set
if test_gen is not None:
    print("\n--- Test Set Evaluation ---")
    results = model.evaluate(test_gen, verbose=1)
    print(f"Test Loss: {results[0]:.4f}")
    print(f"Test Accuracy: {results[1]:.4f} ({results[1]*100:.2f}%)")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

all_preds, all_labels = [], []
for images, labels in test_gen:
    preds = model.predict(images, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_labels.extend(labels)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

class_names = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
from google.colab import files

# Download trained model
files.download('/content/models/efficientnetb0_classifier.h5')

# Download plots
files.download('/content/confusion_matrix.png')
files.download('/content/phase1_history.png')
files.download('/content/phase2_history.png')
files.download('/content/class_distribution.png')
files.download('/content/sample_images.png')
print("All files downloaded")

## Save to Google Drive (Alternative to Download)
Run this cell to save model directly to your Google Drive

In [ ]:
import shutil
import os

drive_save_path = '/content/drive/MyDrive/BrainTumorProject/models/'
os.makedirs(drive_save_path, exist_ok=True)

shutil.copy('/content/models/efficientnetb0_classifier.h5', drive_save_path)
shutil.copy('/content/confusion_matrix.png', drive_save_path)
shutil.copy('/content/phase1_history.png', drive_save_path)
shutil.copy('/content/phase2_history.png', drive_save_path)
print(f"All files saved to Google Drive: {drive_save_path}")